In [1]:
import sys
import os
import json
from glob import glob
import shutil
import random
import ast
import openai
import time
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
# Open JSONL file
dataset_path = "/import/snvm-sc-scratch2/jerrym/dynamic-cheatsheet/data/finlora/train/finer_train_batched.jsonl"
num_samples = 100

samba_client = openai.OpenAI(
    base_url="https://api.sambanova.ai/v1", 
    api_key=""
)

with open(dataset_path, 'r') as json_file:
    all_samples = list(json_file)

random.seed(42)

random.shuffle(all_samples)
print(len(all_samples))
all_samples = all_samples[:num_samples]



10002


In [7]:
def get_embedding(input, embeddings):
    for retry in range(10):
        try: 
            response = samba_client.embeddings.create(
                model="E5-Mistral-7B-Instruct",
                input=[input]
            )
            embeddings.append(response.data[0].embedding)
            break
        except:
            sleep_time = min(2 ** retry, 60)  # Cap at 60 seconds to avoid too long waits
            time.sleep(sleep_time)

In [8]:
question_embeddings = []
for sample in all_samples:
    task_dict = ast.literal_eval(sample)
    all_context  = task_dict["context"]
    index = all_context.index("Answer the following 4 independent questions by providing only")
    context, question = all_context[:index], all_context[index:]
    for retry in range(10):
        try: 
            response = samba_client.embeddings.create(
                model="E5-Mistral-7B-Instruct",
                input=[question]
            )
            question_embeddings.append(response.data[0].embedding)
            break
        except:
            sleep_time = min(2 ** retry, 60)  # Cap at 60 seconds to avoid too long waits
            time.sleep(sleep_time)


print(question_embeddings[0])

[0.0008506526937708259, 0.0014680619351565838, -0.0051587969064712524, 0.004088621120899916, -0.00889069214463234, -0.014488535933196545, -0.012073780409991741, 0.015476390719413757, 0.02294018119573593, 0.005844807252287865, 0.006805221550166607, -0.0022638337686657906, -0.0018796679796651006, 0.005405760370194912, -0.013939728029072285, -0.014817820861935616, 0.0036770147271454334, 0.0022089530248194933, -0.0007340309675782919, -0.01668376848101616, -0.025354938581585884, -0.01229330338537693, 0.011085925623774529, -0.0067229000851511955, 0.010756640695035458, 0.003951418679207563, 0.012732349336147308, -0.009329739026725292, -0.016464244574308395, 0.0022089530248194933, 0.024806130677461624, 0.025245176628232002, 0.024257320910692215, 0.021952327340841293, -0.019208285957574844, -0.010921282693743706, 0.0036495744716376066, -0.008012599311769009, -0.007189386989921331, -0.007079625502228737, -0.005405760370194912, 0.00921997707337141, -0.009165097028017044, -0.001811066991649568, 0.

In [70]:
files = glob("trials_delta/*.txt")
cheatsheet_embeddings = []
for file_path in files:
    with open(file_path, 'r') as f:
        cheatsheet = f.read()
    get_embedding(cheatsheet, cheatsheet_embeddings)


In [79]:
similarity = cosine_similarity(np.array(question_embeddings), np.array(question_embeddings))
np.argsort(similarity[0])[::-1]//4

array([ 0, 16, 23,  9,  2, 11,  1,  9, 24, 20, 10, 15, 13,  3, 24,  8,  5,
        7,  0, 13,  4, 17, 10,  0,  5, 23,  1, 14, 16, 14, 11,  6,  3, 10,
        7, 12,  5,  6,  5,  1,  7, 14,  4, 18, 18, 19, 19, 20,  8, 22, 11,
       20, 22, 18, 13, 16, 13,  6, 18,  3, 21, 17,  3, 15, 22, 15, 11, 23,
        4,  8,  8, 24, 10,  9,  9,  1, 15, 12,  0, 23, 14, 16, 19, 24, 21,
       17, 17, 20,  2,  4,  7, 21, 12, 12,  6, 21,  2, 19, 22,  2])

In [80]:
similarity = cosine_similarity(np.array(cheatsheet_embeddings), np.array(cheatsheet_embeddings))
np.argsort(similarity[0])[::-1]

array([ 0,  7, 23,  3, 12, 15,  6, 21,  4,  9, 19, 17, 20, 13, 14,  8, 10,
        5, 11, 18, 16, 22,  1, 24,  2])

In [2]:
all_embeddings = []
with open("data/finlora/train/finer_train_batched_embeddings.txt", 'r') as f:
    for line in f:
        all_embeddings.append(ast.literal_eval(line))

In [4]:
random.seed(42)
random.shuffle(all_embeddings)
with open("data/finlora/train/finer_train_batched_embeddings_500.txt", 'w+') as f:
    for embed in all_embeddings[:500]:
        f.write(str(embed) + "\n")

In [5]:
embeddings_500 = []
with open("data/finlora/train/finer_train_batched_embeddings_500.txt", 'r') as f:
    for line in f:
        embeddings_500.append(ast.literal_eval(line))
print(embeddings_500[0])

[0.0008506526937708259, 0.0014680619351565838, -0.0051587969064712524, 0.004088621120899916, -0.00889069214463234, -0.014488535933196545, -0.012073780409991741, 0.015476390719413757, 0.02294018119573593, 0.005844807252287865, 0.006805221550166607, -0.0022638337686657906, -0.0018796679796651006, 0.005405760370194912, -0.013939728029072285, -0.014817820861935616, 0.0036770147271454334, 0.0022089530248194933, -0.0007340309675782919, -0.01668376848101616, -0.025354938581585884, -0.01229330338537693, 0.011085925623774529, -0.0067229000851511955, 0.010756640695035458, 0.003951418679207563, 0.012732349336147308, -0.009329739026725292, -0.016464244574308395, 0.0022089530248194933, 0.024806130677461624, 0.025245176628232002, 0.024257320910692215, 0.021952327340841293, -0.019208285957574844, -0.010921282693743706, 0.0036495744716376066, -0.008012599311769009, -0.007189386989921331, -0.007079625502228737, -0.005405760370194912, 0.00921997707337141, -0.009165097028017044, -0.001811066991649568, 0.